<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day12_a_(260324)_%EA%B3%84%EC%A2%8C%EC%9D%B4%EC%B2%B4_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile /content/server.js

const express = require('express');
const path = require('path');
const mysql = require('mysql2/promise');
const jwt = require('jsonwebtoken');
const bcrypt = require('bcrypt');

const app = express();
const PORT = 3000;
const JWT_SECRET = 'my_super_secret_key'; // 클라이언트와 서버가 공유하는 비밀 열쇠

app.use(express.json());
app.use('/static', express.static(path.join(__dirname, 'static')));

// MariaDB 연결 설정
const pool = mysql.createPool({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb'
});

// 계좌번호 생성 함수
function makeAccountNumber() {
    const now = Date.now().toString().slice(-8);
    const rand = Math.floor(Math.random() * 9000) + 1000;
    return `1002-${now}-${rand}`;
}

// JWT 확인 함수
function verifyToken(req) {
    const authHeader = req.headers['authorization'];
    const token = authHeader && authHeader.split(' ')[1];
    if (!token) return null;

    try {
        return jwt.verify(token, JWT_SECRET);
    } catch (err) {
        return null;
    }
}

// [1] 페이지 라우팅: templates 폴더 안의 파일들을 읽어옵니다.
app.get('/', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'login.html')));
app.get('/signup', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'signup.html')));
app.get('/main', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'main.html')));

// [2] API: 회원가입 (비밀번호 암호화 후 DB 저장 + 계좌 자동 생성)
app.post('/api/signup', async (req, res) => {
    let conn;
    try {
        const { userId, password, userName, role } = req.body;

        if (!userId || !password || !userName) {
            return res.status(400).json({ success: false, message: '입력값 누락' });
        }

        const userRole = role === 'admin' ? 'admin' : 'user';
        const hashed = await bcrypt.hash(password, 10);
        const accountNumber = makeAccountNumber();

        conn = await pool.getConnection();
        await conn.beginTransaction();

        await conn.execute(
            'INSERT INTO users (user_id, password, user_name, role) VALUES (?, ?, ?, ?)',
            [userId, hashed, userName, userRole]
        );

        await conn.execute(
            'INSERT INTO accounts (user_id, account_number, balance) VALUES (?, ?, ?)',
            [userId, accountNumber, 100000]
        );

        await conn.execute(
            `INSERT INTO account_transactions
            (account_number, tx_type, amount, balance_after, related_account, transfer_id, memo)
            VALUES (?, 'DEPOSIT', ?, ?, NULL, NULL, '초기입금')`,
            [accountNumber, 100000, 100000]
        );

        await conn.commit();
        res.json({ success: true, accountNumber });

    } catch (err) {
        if (conn) await conn.rollback();
        console.error(err);
        res.status(500).json({ success: false, message: '회원가입 실패' });
    } finally {
        if (conn) conn.release();
    }
});

// [3] API: 로그인 (검증 성공 시 JWT 발급)
app.post('/api/login', async (req, res) => {
    try {
        const { userId, password } = req.body;
        const [rows] = await pool.execute('SELECT * FROM users WHERE user_id = ?', [userId]);
        const user = rows[0];

        if (user && await bcrypt.compare(password, user.password)) {
            const token = jwt.sign(
                {
                    userId: user.user_id,
                    userName: user.user_name,
                    role: user.role
                },
                JWT_SECRET,
                { expiresIn: '1h' }
            );

            res.json({ success: true, token });
        } else {
            res.status(401).json({ success: false, message: '로그인 실패' });
        }
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false, message: '서버 오류' });
    }
});

// [4] API: JWT 검증 (클라이언트가 보낸 토큰이 진짜인지 확인)
app.get('/api/verify', (req, res) => {
    const authHeader = req.headers['authorization'];
    const token = authHeader && authHeader.split(' ')[1];

    if (!token) return res.json({ success: false });

    jwt.verify(token, JWT_SECRET, (err, decoded) => {
        if (err) return res.json({ success: false });
        res.json({ success: true, user: decoded });
    });
});

// [5] API: 보호된 데이터 조회
app.get('/api/data', async (req, res) => {
    try {
        const decoded = verifyToken(req);
        if (!decoded) {
            return res.status(401).json({ success: false, message: '인증 실패' });
        }

        if (decoded.role === 'admin') {
            const [users] = await pool.execute(`
                SELECT u.user_id, u.user_name, u.role, u.created_at, a.account_number, a.balance
                FROM users u
                LEFT JOIN accounts a ON u.user_id = a.user_id
                ORDER BY u.created_at DESC, a.account_number ASC
            `);

            const [transfers] = await pool.execute(`
                SELECT *
                FROM transfers
                ORDER BY transferred_at DESC
            `);

            const [transactions] = await pool.execute(`
                SELECT *
                FROM account_transactions
                ORDER BY created_at DESC, tx_id DESC
            `);

            return res.json({
                success: true,
                role: 'admin',
                users,
                transfers,
                transactions
            });
        } else {
            const [accounts] = await pool.execute(
                'SELECT * FROM accounts WHERE user_id = ? ORDER BY created_at DESC',
                [decoded.userId]
            );

            const [transactions] = await pool.execute(`
                SELECT at.*
                FROM account_transactions at
                JOIN accounts a ON at.account_number = a.account_number
                WHERE a.user_id = ?
                ORDER BY at.created_at DESC, at.tx_id DESC
            `, [decoded.userId]);

            return res.json({
                success: true,
                role: 'user',
                user: decoded,
                accounts,
                transactions
            });
        }
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false, message: '데이터 조회 실패' });
    }
});

// [6] API: 계좌이체
app.post('/api/transfer', async (req, res) => {
    let conn;
    try {
        const decoded = verifyToken(req);
        if (!decoded) {
            return res.status(401).json({ success: false, message: '인증 실패' });
        }

        if (decoded.role !== 'user') {
            return res.status(403).json({ success: false, message: '사용자만 이체 가능' });
        }

        const { fromAccountNumber, toAccountNumber, amount, memo } = req.body;
        const sendAmount = Number(amount);

        if (!fromAccountNumber || !toAccountNumber || !sendAmount) {
            return res.status(400).json({ success: false, message: '입력값 누락' });
        }

        if (fromAccountNumber === toAccountNumber) {
            return res.status(400).json({ success: false, message: '동일 계좌 이체 불가' });
        }

        if (sendAmount <= 0) {
            return res.status(400).json({ success: false, message: '금액 오류' });
        }

        conn = await pool.getConnection();
        await conn.beginTransaction();

        const [fromRows] = await conn.execute(
            'SELECT * FROM accounts WHERE account_number = ? FOR UPDATE',
            [fromAccountNumber]
        );
        const fromAccount = fromRows[0];

        const [toRows] = await conn.execute(
            'SELECT * FROM accounts WHERE account_number = ? FOR UPDATE',
            [toAccountNumber]
        );
        const toAccount = toRows[0];

        if (!fromAccount) {
            await conn.rollback();
            return res.status(404).json({ success: false, message: '출금 계좌 없음' });
        }

        if (!toAccount) {
            await conn.rollback();
            return res.status(404).json({ success: false, message: '입금 계좌 없음' });
        }

        if (fromAccount.user_id !== decoded.userId) {
            await conn.rollback();
            return res.status(403).json({ success: false, message: '본인 계좌만 출금 가능' });
        }

        if (fromAccount.balance < sendAmount) {
            await conn.rollback();
            return res.status(400).json({ success: false, message: '잔액 부족' });
        }

        const newFromBalance = fromAccount.balance - sendAmount;
        const newToBalance = toAccount.balance + sendAmount;

        await conn.execute(
            'UPDATE accounts SET balance = ? WHERE account_number = ?',
            [newFromBalance, fromAccountNumber]
        );

        await conn.execute(
            'UPDATE accounts SET balance = ? WHERE account_number = ?',
            [newToBalance, toAccountNumber]
        );

        const [transferResult] = await conn.execute(`
            INSERT INTO transfers
            (from_account_number, to_account_number, sender_user_id, receiver_user_id, amount, transfer_memo)
            VALUES (?, ?, ?, ?, ?, ?)
        `, [
            fromAccountNumber,
            toAccountNumber,
            fromAccount.user_id,
            toAccount.user_id,
            sendAmount,
            memo || ''
        ]);

        const transferId = transferResult.insertId;

        await conn.execute(`
            INSERT INTO account_transactions
            (account_number, tx_type, amount, balance_after, related_account, transfer_id, memo)
            VALUES (?, 'TRANSFER_OUT', ?, ?, ?, ?, ?)
        `, [
            fromAccountNumber,
            sendAmount,
            newFromBalance,
            toAccountNumber,
            transferId,
            memo || ''
        ]);

        await conn.execute(`
            INSERT INTO account_transactions
            (account_number, tx_type, amount, balance_after, related_account, transfer_id, memo)
            VALUES (?, 'TRANSFER_IN', ?, ?, ?, ?, ?)
        `, [
            toAccountNumber,
            sendAmount,
            newToBalance,
            fromAccountNumber,
            transferId,
            memo || ''
        ]);

        await conn.commit();
        res.json({ success: true, message: '이체 완료' });

    } catch (err) {
        if (conn) await conn.rollback();
        console.error(err);
        res.status(500).json({ success: false, message: '이체 실패' });
    } finally {
        if (conn) conn.release();
    }
});

// [7] API: 관리자 - 사용자 강제 추가
app.post('/api/admin/add-user', async (req, res) => {
    let conn;
    try {
        const decoded = verifyToken(req);
        if (!decoded) {
            return res.status(401).json({ success: false, message: '인증 실패' });
        }

        if (decoded.role !== 'admin') {
            return res.status(403).json({ success: false, message: '관리자만 가능' });
        }

        const { userId, password, userName, role } = req.body;

        if (!userId || !password || !userName) {
            return res.status(400).json({ success: false, message: '입력값 누락' });
        }

        const userRole = role === 'admin' ? 'admin' : 'user';
        const hashed = await bcrypt.hash(password, 10);

        conn = await pool.getConnection();
        await conn.beginTransaction();

        await conn.execute(
            'INSERT INTO users (user_id, password, user_name, role) VALUES (?, ?, ?, ?)',
            [userId, hashed, userName, userRole]
        );

        await conn.commit();
        res.json({ success: true, message: '사용자 추가 완료' });

    } catch (err) {
        if (conn) await conn.rollback();
        console.error(err);
        res.status(500).json({ success: false, message: '사용자 추가 실패' });
    } finally {
        if (conn) conn.release();
    }
});

// [8] API: 관리자 - 계좌 강제 추가
app.post('/api/admin/add-account', async (req, res) => {
    let conn;
    try {
        const decoded = verifyToken(req);
        if (!decoded) {
            return res.status(401).json({ success: false, message: '인증 실패' });
        }

        if (decoded.role !== 'admin') {
            return res.status(403).json({ success: false, message: '관리자만 가능' });
        }

        const { userId, accountNumber, balance } = req.body;
        const firstBalance = Number(balance) || 0;

        if (!userId || !accountNumber) {
            return res.status(400).json({ success: false, message: '입력값 누락' });
        }

        conn = await pool.getConnection();
        await conn.beginTransaction();

        const [userRows] = await conn.execute(
            'SELECT * FROM users WHERE user_id = ?',
            [userId]
        );

        if (userRows.length === 0) {
            await conn.rollback();
            return res.status(404).json({ success: false, message: '사용자 없음' });
        }

        await conn.execute(
            'INSERT INTO accounts (user_id, account_number, balance) VALUES (?, ?, ?)',
            [userId, accountNumber, firstBalance]
        );

        await conn.execute(`
            INSERT INTO account_transactions
            (account_number, tx_type, amount, balance_after, related_account, transfer_id, memo)
            VALUES (?, 'DEPOSIT', ?, ?, NULL, NULL, '관리자 강제 계좌 생성')
        `, [accountNumber, firstBalance, firstBalance]);

        await conn.commit();
        res.json({ success: true, message: '계좌 추가 완료' });

    } catch (err) {
        if (conn) await conn.rollback();
        console.error(err);
        res.status(500).json({ success: false, message: '계좌 추가 실패' });
    } finally {
        if (conn) conn.release();
    }
});

// [9] API: 관리자 - 사용자 비밀번호 수정
app.post('/api/admin/reset-password', async (req, res) => {
    let conn;
    try {
        const decoded = verifyToken(req);
        if (!decoded) {
            return res.status(401).json({ success: false, message: '인증 실패' });
        }

        if (decoded.role !== 'admin') {
            return res.status(403).json({ success: false, message: '관리자만 가능' });
        }

        const { userId, newPassword } = req.body;

        if (!userId || !newPassword) {
            return res.status(400).json({ success: false, message: '입력값 누락' });
        }

        const hashed = await bcrypt.hash(newPassword, 10);

        conn = await pool.getConnection();
        await conn.beginTransaction();

        const [rows] = await conn.execute(
            'SELECT * FROM users WHERE user_id = ?',
            [userId]
        );

        if (rows.length === 0) {
            await conn.rollback();
            return res.status(404).json({ success: false, message: '사용자 없음' });
        }

        await conn.execute(
            'UPDATE users SET password = ? WHERE user_id = ?',
            [hashed, userId]
        );

        await conn.commit();

        res.json({ success: true, message: '비밀번호 수정 완료' });

    } catch (err) {
        if (conn) await conn.rollback();
        console.error(err);
        res.status(500).json({ success: false, message: '비밀번호 수정 실패' });
    } finally {
        if (conn) conn.release();
    }
});

app.listen(PORT, () => console.log(`서버 실행 중: http://localhost:${PORT}`));

Writing /content/server.js


In [2]:
%%writefile /content/templates/signup.html

<h1>회원가입</h1>
<input type="text" id="uId" placeholder="아이디"><br>
<input type="password" id="uPw" placeholder="비밀번호"><br>
<input type="text" id="uName" placeholder="이름"><br>

<select id="uRole">
    <option value="user">사용자</option>
    <option value="admin">관리자</option>
</select><br><br>

<button onclick="save()">가입하기</button>

<script>
    async function save() {
        const res = await fetch('/api/signup', {
            method: 'POST',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({
                userId: uId.value,
                password: uPw.value,
                userName: uName.value,
                role: uRole.value
            })
        });

        const data = await res.json();

        if (data.success) {
            alert('가입 완료\n생성된 계좌번호: ' + data.accountNumber);
            location.href = '/';
        } else {
            alert(data.message || '가입 실패');
        }
    }
</script>

Writing /content/templates/signup.html


In [3]:
%%writefile /content/templates/login.html

<h1>로그인</h1>

<input type="text" id="uId" placeholder="아이디"><br>
<input type="password" id="uPw" placeholder="비밀번호"><br>

<button onclick="login()">로그인</button>
<button onclick="location.href='/signup'">회원가입</button>

<script>
    async function login() {
        const res = await fetch('/api/login', {
            method: 'POST',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({userId: uId.value, password: uPw.value})
        });

        const data = await res.json();

        if(data.success) {
            localStorage.setItem('myToken', data.token);
            location.href = '/main';
        } else {
            alert(data.message || '로그인 실패');
        }
    }
</script>

Writing /content/templates/login.html


In [4]:
%%writefile /content/templates/main.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>토스뱅크</title>
    <link rel="stylesheet" href="/static/css/style.css">
</head>
<body>

<h1>메인 대시보드</h1>
<p id="welcome"></p>
<p id="roleInfo"></p>

<button id="accountBtn" onclick="reloadAccount()">계좌조회</button>
<button id="transferBtn" onclick="showMenu('transfer')">계좌이체</button>
<button onclick="logout()">로그아웃</button>

<hr>

<div id="accountSection" style="display:none;">
    <h3>계좌조회</h3>
    <ul id="accountList"></ul>

    <h3>거래 내역</h3>
    <table border="1" width="100%">
        <thead>
            <tr>
                <th>일시</th>
                <th>구분</th>
                <th>금액</th>
                <th>상대계좌</th>
                <th>잔액</th>
                <th>메모</th>
            </tr>
        </thead>
        <tbody id="transactionTable"></tbody>
    </table>
</div>

<div id="transferSection" style="display:none;">
    <h3>계좌이체</h3>
    <input type="text" id="fromAccountNumber" placeholder="내 출금 계좌번호"><br>
    <input type="text" id="toAccountNumber" placeholder="받는 계좌번호"><br>
    <input type="number" id="amount" placeholder="이체 금액"><br>
    <input type="text" id="memo" placeholder="메모"><br>
    <button onclick="sendTransfer()">이체하기</button>
</div>

<div id="adminSection" style="display:none;">
    <h3>관리자 기능</h3>

    <h4>전체 사용자 및 계좌 목록</h4>
    <ul id="adminUserList"></ul>

    <h4>전체 이체 내역</h4>
    <ul id="adminTransferList"></ul>

    <hr>

    <h4>사용자 강제 추가</h4>
    <input type="text" id="adminUserId" placeholder="아이디"><br>
    <input type="password" id="adminUserPw" placeholder="비밀번호"><br>
    <input type="text" id="adminUserName" placeholder="이름"><br>
    <select id="adminUserRole">
        <option value="user">사용자</option>
        <option value="admin">관리자</option>
    </select><br>
    <button onclick="addUserByAdmin()">사용자 추가</button>

    <hr>

    <h4>계좌 강제 추가</h4>
    <input type="text" id="adminAccountUserId" placeholder="사용자 아이디"><br>
    <input type="text" id="adminAccountNumber" placeholder="계좌번호"><br>
    <input type="number" id="adminAccountBalance" placeholder="초기 잔액"><br>
    <button onclick="addAccountByAdmin()">계좌 추가</button>

    <hr>

    <h4>비밀번호 초기화</h4>
    <input type="text" id="resetUserId" placeholder="사용자 아이디"><br>
    <input type="password" id="resetPasswordInput" placeholder="새 비밀번호"><br>
    <button onclick="resetUserPassword()">비밀번호 변경</button>
</div>

<script>
    let loginUser = null;
    let refreshTimer = null;
    let isLoading = false;
    let needReload = false;
    let currentMenu = 'account';

    function logout() {
        stopAutoRefresh();
        localStorage.removeItem('myToken');
        location.href = '/';
    }

    function startAutoRefresh() {
        stopAutoRefresh();
        refreshTimer = setInterval(async () => {
            if (currentMenu === 'account') {
                await loadData();
            }
        }, 3000);
    }

    function stopAutoRefresh() {
        if (refreshTimer) {
            clearInterval(refreshTimer);
            refreshTimer = null;
        }
    }

    async function init() {
        const token = localStorage.getItem('myToken');
        if (!token) {
            location.href = '/';
            return;
        }

        const authRes = await fetch('/api/verify', {
            headers: {
                'Authorization': `Bearer ${token}`,
                'Cache-Control': 'no-cache'
            },
            cache: 'no-store'
        });

        const authData = await authRes.json();

        if (authData.success) {
            loginUser = authData.user;
            document.getElementById('welcome').innerText = authData.user.userName + '님 반갑습니다.';
            document.getElementById('roleInfo').innerText = '권한: ' + authData.user.role;

            await loadData();
            showMenu('account');
            startAutoRefresh();
        } else {
            alert('인증이 만료되었습니다.');
            localStorage.removeItem('myToken');
            location.href = '/';
        }
    }

    function showMenu(menu) {
        currentMenu = menu;

        document.getElementById('accountSection').style.display = 'none';
        document.getElementById('transferSection').style.display = 'none';

        if (menu === 'account') {
            document.getElementById('accountSection').style.display = 'block';
        }

        if (menu === 'transfer') {
            if (loginUser && loginUser.role === 'admin') {
                alert('관리자는 계좌이체 메뉴를 사용할 수 없습니다.');
                currentMenu = 'account';
                document.getElementById('accountSection').style.display = 'block';
                return;
            }
            document.getElementById('transferSection').style.display = 'block';
        }
    }

    async function reloadAccount() {
        await loadData();
        showMenu('account');
    }

    function typeToKorean(type) {
        if (type === 'DEPOSIT') return '입금';
        if (type === 'WITHDRAW') return '출금';
        if (type === 'TRANSFER_OUT') return '이체출금';
        if (type === 'TRANSFER_IN') return '이체입금';
        return type;
    }

    function formatNumber(value) {
        return Number(value).toLocaleString('ko-KR');
    }

    async function loadData() {
        if (isLoading) {
            needReload = true;
            return;
        }

        isLoading = true;

        try {
            const token = localStorage.getItem('myToken');

            const res = await fetch('/api/data', {
                headers: {
                    'Authorization': `Bearer ${token}`,
                    'Cache-Control': 'no-cache'
                },
                cache: 'no-store'
            });

            const data = await res.json();

            if (!data.success) {
                alert(data.message || '데이터 조회 실패');
                return;
            }

            const transactionTable = document.getElementById('transactionTable');
            transactionTable.innerHTML = '';

            const accountList = document.getElementById('accountList');
            accountList.innerHTML = '';

            if (data.role === 'user') {
                document.getElementById('adminSection').style.display = 'none';

                data.accounts.forEach(item => {
                    const li = document.createElement('li');
                    li.innerText = `계좌번호: ${item.account_number} / 잔액: ${formatNumber(item.balance)}원`;
                    accountList.appendChild(li);
                });

                if (data.accounts.length > 0) {
                    document.getElementById('fromAccountNumber').value = data.accounts[0].account_number;
                }

                if (data.transactions && data.transactions.length > 0) {
                    data.transactions.forEach(item => {
                        const tr = document.createElement('tr');
                        tr.innerHTML = `
                            <td>${item.created_at}</td>
                            <td>${typeToKorean(item.tx_type)}</td>
                            <td>${formatNumber(item.amount)}원</td>
                            <td>${item.related_account || '-'}</td>
                            <td>${formatNumber(item.balance_after)}원</td>
                            <td>${item.memo || ''}</td>
                        `;
                        transactionTable.appendChild(tr);
                    });
                }

            } else if (data.role === 'admin') {
                document.getElementById('adminSection').style.display = 'block';

                data.users.forEach(item => {
                    const li = document.createElement('li');
                    li.innerText = `아이디: ${item.user_id} / 이름: ${item.user_name} / 계좌번호: ${item.account_number || '-'} / 잔액: ${formatNumber(item.balance ?? 0)}원`;
                    accountList.appendChild(li);
                });

                if (data.transactions && data.transactions.length > 0) {
                    data.transactions.forEach(item => {
                        const tr = document.createElement('tr');
                        tr.innerHTML = `
                            <td>${item.created_at}</td>
                            <td>${typeToKorean(item.tx_type)}</td>
                            <td>${formatNumber(item.amount)}원</td>
                            <td>${item.related_account || '-'}</td>
                            <td>${formatNumber(item.balance_after)}원</td>
                            <td>${item.memo || ''}</td>
                        `;
                        transactionTable.appendChild(tr);
                    });
                }

                const adminUserList = document.getElementById('adminUserList');
                adminUserList.innerHTML = '';

                data.users.forEach(item => {
                    const li = document.createElement('li');
                    li.innerText =
                        `아이디: ${item.user_id} / 이름: ${item.user_name} / 권한: ${item.role} / 계좌번호: ${item.account_number || '-'} / 잔액: ${formatNumber(item.balance ?? 0)}원`;
                    adminUserList.appendChild(li);
                });

                const adminTransferList = document.getElementById('adminTransferList');
                adminTransferList.innerHTML = '';

                if (data.transfers && data.transfers.length > 0) {
                    data.transfers.forEach(item => {
                        const li = document.createElement('li');
                        li.innerText =
                            `${item.transferred_at} / ${item.sender_user_id} → ${item.receiver_user_id} / ${item.from_account_number} → ${item.to_account_number} / ${formatNumber(item.amount)}원 / ${item.transfer_memo || ''}`;
                        adminTransferList.appendChild(li);
                    });
                }
            }
        } finally {
            isLoading = false;

            if (needReload) {
                needReload = false;
                await loadData();
            }
        }
    }

    async function sendTransfer() {
        const token = localStorage.getItem('myToken');

        const res = await fetch('/api/transfer', {
            method: 'POST',
            headers: {
                'Content-Type': 'application/json',
                'Authorization': `Bearer ${token}`,
                'Cache-Control': 'no-cache'
            },
            cache: 'no-store',
            body: JSON.stringify({
                fromAccountNumber: document.getElementById('fromAccountNumber').value,
                toAccountNumber: document.getElementById('toAccountNumber').value,
                amount: document.getElementById('amount').value,
                memo: document.getElementById('memo').value
            })
        });

        const data = await res.json();

        if (data.success) {
            alert('이체 완료');
            document.getElementById('toAccountNumber').value = '';
            document.getElementById('amount').value = '';
            document.getElementById('memo').value = '';
            await loadData();
            showMenu('account');
        } else {
            alert(data.message || '이체 실패');
        }
    }

    async function addUserByAdmin() {
        const token = localStorage.getItem('myToken');

        const res = await fetch('/api/admin/add-user', {
            method: 'POST',
            headers: {
                'Content-Type': 'application/json',
                'Authorization': `Bearer ${token}`,
                'Cache-Control': 'no-cache'
            },
            cache: 'no-store',
            body: JSON.stringify({
                userId: document.getElementById('adminUserId').value,
                password: document.getElementById('adminUserPw').value,
                userName: document.getElementById('adminUserName').value,
                role: document.getElementById('adminUserRole').value
            })
        });

        const data = await res.json();

        if (data.success) {
            alert('사용자 추가 완료');
            document.getElementById('adminUserId').value = '';
            document.getElementById('adminUserPw').value = '';
            document.getElementById('adminUserName').value = '';
            await loadData();
            showMenu('account');
        } else {
            alert(data.message || '사용자 추가 실패');
        }
    }

    async function addAccountByAdmin() {
        const token = localStorage.getItem('myToken');

        const res = await fetch('/api/admin/add-account', {
            method: 'POST',
            headers: {
                'Content-Type': 'application/json',
                'Authorization': `Bearer ${token}`,
                'Cache-Control': 'no-cache'
            },
            cache: 'no-store',
            body: JSON.stringify({
                userId: document.getElementById('adminAccountUserId').value,
                accountNumber: document.getElementById('adminAccountNumber').value,
                balance: document.getElementById('adminAccountBalance').value
            })
        });

        const data = await res.json();

        if (data.success) {
            alert('계좌 추가 완료');
            document.getElementById('adminAccountUserId').value = '';
            document.getElementById('adminAccountNumber').value = '';
            document.getElementById('adminAccountBalance').value = '';
            await loadData();
            showMenu('account');
        } else {
            alert(data.message || '계좌 추가 실패');
        }
    }

    async function resetUserPassword() {
        const token = localStorage.getItem('myToken');

        const res = await fetch('/api/admin/reset-password', {
            method: 'POST',
            headers: {
                'Content-Type': 'application/json',
                'Authorization': `Bearer ${token}`,
                'Cache-Control': 'no-cache'
            },
            cache: 'no-store',
            body: JSON.stringify({
                userId: document.getElementById('resetUserId').value,
                newPassword: document.getElementById('resetPasswordInput').value
            })
        });

        const data = await res.json();

        if (data.success) {
            alert('비밀번호 변경 완료');
            document.getElementById('resetUserId').value = '';
            document.getElementById('resetPasswordInput').value = '';
        } else {
            alert(data.message || '비밀번호 변경 실패');
        }
    }

    window.addEventListener('beforeunload', stopAutoRefresh);

    init();
</script>

</body>
</html>

Writing /content/templates/main.html


In [6]:
%%writefile /content/static/css/style.css

@import url("https://cdn.jsdelivr.net/gh/orioncactus/pretendard@v1.3.9/dist/web/variable/pretendardvariable.min.css");

/* 기본 초기화 */
* {
    box-sizing: border-box;
    margin: 0;
    padding: 0;
}

body {
    font-family: "Pretendard Variable", Pretendard, system-ui;
    background-color: #f6f7f9;
    color: #222;
    padding: 20px;
}

/* 제목 */
h1 {
    font-size: 26px;
    font-weight: 700;
    margin-bottom: 10px;
}

h3 {
    margin-top: 20px;
    margin-bottom: 10px;
    font-weight: 600;
}

/* 사용자 정보 */
#welcome {
    font-size: 16px;
    margin-bottom: 5px;
}

#roleInfo {
    font-size: 14px;
    color: #666;
    margin-bottom: 15px;
}

/* 버튼 */
button {
    padding: 10px 16px;
    margin-right: 6px;
    margin-bottom: 10px;
    border: none;
    border-radius: 8px;
    background-color: #3182f6;
    color: white;
    font-size: 14px;
    cursor: pointer;
    transition: 0.2s;
}

button:hover {
    background-color: #1b64da;
}

/* 입력 */
input, select {
    width: 260px;
    padding: 10px;
    margin-bottom: 8px;
    border: 1px solid #ddd;
    border-radius: 8px;
    font-size: 14px;
}

/* 카드 영역 */
#accountSection,
#transferSection,
#adminSection {
    background: white;
    padding: 20px;
    border-radius: 14px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.05);
    margin-top: 15px;
}

/* 리스트 */
ul {
    list-style: none;
    margin-bottom: 10px;
}

ul li {
    padding: 8px 0;
    border-bottom: 1px solid #eee;
    font-size: 14px;
}

/* 테이블 */
table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 10px;
    background: white;
}

thead {
    background-color: #f1f3f5;
}

th {
    padding: 10px;
    font-size: 13px;
    text-align: center;
    border-bottom: 1px solid #ddd;
}

td {
    padding: 10px;
    font-size: 13px;
    text-align: center;
    border-bottom: 1px solid #eee;
}

/* 거래 색상 */
td:nth-child(3) {
    font-weight: 600;
}

tr:hover {
    background-color: #f9fafb;
}

/* 구분 컬러 */
td:nth-child(2) {
    font-weight: 600;
}

td:nth-child(2):contains("입금") {
    color: #2e7d32;
}

td:nth-child(2):contains("출금") {
    color: #c62828;
}

/* 구분 색상 보정 (JS에서 class 주면 더 정확함) */
.deposit { color: #2e7d32; }
.withdraw { color: #c62828; }
.transfer-in { color: #1565c0; }
.transfer-out { color: #ef6c00; }

/* 반응형 */
@media (max-width: 768px) {
    input, select {
        width: 100%;
    }

    button {
        width: 100%;
        margin-bottom: 6px;
    }

    table {
        font-size: 12px;
    }
}

Writing /content/static/css/style.css


In [7]:
!npm install express mysql2 jsonwebtoken bcrypt

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 93 packages in 9s
⠹
⠹25 packages are looking for funding
⠹  run `npm fund` for details
⠹

In [8]:
!nohup node server.js > server.log 2>&1 &

In [9]:
!npm install -g cloudflared

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 1 package in 2s
⠇

In [10]:
!nohup cloudflared tunnel --url http://localhost:3000 > tunnel.log 2>&1 &

In [11]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" tunnel.log |tail -n 1

https://personals-perth-boundary-considered.trycloudflare.com
